# 🍇 Grapes Price Predictor — Machine Learning Project
**Course:** Machine Learning | **Task:** TEST2 | **Context:** Tanzanian Grapes Market

This notebook covers: Data generation & preprocessing, Linear Regression & Decision Tree training, Model evaluation & comparison, Visualizations, and saving the best model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import pickle, warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully!')

## 1. Dataset Generation (Tanzanian Grapes Market)

In [ ]:
np.random.seed(42)
n = 500
regions     = ['Dodoma','Arusha','Morogoro','Mbeya','Singida']
grape_types = ['Red Globe','Thompson Seedless','Muscat','Flame Seedless']
seasons     = ['Dry Season','Wet Season']
qualities   = ['Grade A','Grade B','Grade C']

region_col     = np.random.choice(regions, n)
grape_type_col = np.random.choice(grape_types, n)
season_col     = np.random.choice(seasons, n)
quality_col    = np.random.choice(qualities, n)
weight_kg      = np.round(np.random.uniform(0.5, 10.0, n), 2)
sugar_content  = np.round(np.random.uniform(10.0, 25.0, n), 1)
farm_size_ha   = np.round(np.random.uniform(0.5, 20.0, n), 1)
distance_km    = np.round(np.random.uniform(5, 300, n), 1)
rainfall_mm    = np.round(np.random.uniform(400, 1200, n), 1)

price = (
    2500
    + sugar_content * 80
    + weight_kg * 50
    - distance_km * 3
    + farm_size_ha * 20
    + rainfall_mm * 0.5
    + np.where(quality_col=='Grade A', 800, np.where(quality_col=='Grade B', 300, 0))
    + np.where(season_col=='Dry Season', 400, -200)
    + np.where(grape_type_col=='Muscat', 600, np.where(grape_type_col=='Red Globe', 300, 0))
    + np.random.normal(0, 300, n)
)
price = np.round(np.clip(price, 1000, 12000), 2)

df = pd.DataFrame({
    'Region': region_col, 'Grape_Type': grape_type_col, 'Season': season_col,
    'Quality_Grade': quality_col, 'Weight_kg': weight_kg,
    'Sugar_Content_Brix': sugar_content, 'Farm_Size_ha': farm_size_ha,
    'Distance_to_Market_km': distance_km, 'Rainfall_mm': rainfall_mm,
    'Price_TZS_per_kg': price
})
df.to_csv('grapes_dataset.csv', index=False)
print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} cols')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(df.describe())
print('\nMissing:', df.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle('Grapes Price Analysis — Tanzania', fontsize=16, fontweight='bold', color='purple')

axes[0,0].hist(df['Price_TZS_per_kg'], bins=30, color='purple', alpha=0.7, edgecolor='white')
axes[0,0].set_title('Price Distribution (TZS/kg)'); axes[0,0].set_xlabel('Price'); axes[0,0].set_ylabel('Frequency')

df.groupby('Grape_Type')['Price_TZS_per_kg'].mean().sort_values().plot(kind='barh', ax=axes[0,1], color='mediumorchid')
axes[0,1].set_title('Avg Price by Grape Type'); axes[0,1].set_xlabel('TZS/kg')

axes[1,0].scatter(df['Sugar_Content_Brix'], df['Price_TZS_per_kg'], alpha=0.4, color='darkviolet', s=20)
axes[1,0].set_title('Sugar Content vs Price'); axes[1,0].set_xlabel('Brix %'); axes[1,0].set_ylabel('Price')

df.groupby('Quality_Grade')['Price_TZS_per_kg'].mean().plot(kind='bar', ax=axes[1,1], color=['gold','silver','#cd7f32'])
axes[1,1].set_title('Avg Price by Quality'); axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight'); plt.show()
print('EDA plots saved.')

## 3. Data Preprocessing

In [ ]:
df_encoded = df.copy()
le = LabelEncoder()
cat_cols = ['Region','Grape_Type','Season','Quality_Grade']
encoders = {}
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le

X = df_encoded.drop('Price_TZS_per_kg', axis=1)
y = df_encoded['Price_TZS_per_kg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Features: {list(X.columns)}')

## 4. Model Training

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

dt_model = DecisionTreeRegressor(max_depth=8, min_samples_split=10, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
print('Both models trained!')

## 5. Model Evaluation & Comparison

In [ ]:
def evaluate(name, y_true, y_pred):
    return {'Model': name,
            'MAE': round(mean_absolute_error(y_true, y_pred),2),
            'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)),2),
            'R2_Score': round(r2_score(y_true, y_pred),4)}

results_df = pd.DataFrame([evaluate('Linear Regression', y_test, lr_preds),
                           evaluate('Decision Tree', y_test, dt_preds)])
print(results_df.to_string(index=False))
best = results_df.loc[results_df['R2_Score'].idxmax(), 'Model']
print(f'\nBest Model: {best}')

## 6. Visualizations

In [ ]:
lr_m = evaluate('Linear Regression', y_test, lr_preds)
dt_m = evaluate('Decision Tree', y_test, dt_preds)

fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.suptitle('Model Evaluation — Grapes Price Predictor', fontsize=15, fontweight='bold')

x = np.arange(2); w = 0.35
axes[0].bar(x-w/2,[lr_m['MAE'],lr_m['RMSE']],w,label='Linear Regression',color='steelblue')
axes[0].bar(x+w/2,[dt_m['MAE'],dt_m['RMSE']],w,label='Decision Tree',color='mediumorchid')
axes[0].set_xticks(x); axes[0].set_xticklabels(['MAE','RMSE'])
axes[0].set_title('Error Metrics'); axes[0].legend()

axes[1].bar(['Linear Regression','Decision Tree'],[lr_m['R2_Score'],dt_m['R2_Score']],
            color=['steelblue','mediumorchid'])
axes[1].set_title('R² Score'); axes[1].set_ylim(0,1)
for i,v in enumerate([lr_m['R2_Score'],dt_m['R2_Score']]):
    axes[1].text(i, v+0.01, str(v), ha='center', fontweight='bold')

best_preds = dt_preds if best=='Decision Tree' else lr_preds
axes[2].scatter(y_test, best_preds, alpha=0.4, color='darkviolet', s=20)
mn,mx = y_test.min(), y_test.max()
axes[2].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect')
axes[2].set_title(f'Actual vs Predicted ({best})')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].legend()

plt.tight_layout(); plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
feat_imp = pd.Series(dt_model.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8,5))
feat_imp.plot(kind='barh', ax=ax, color='mediumorchid', edgecolor='white')
ax.set_title('Feature Importances — Decision Tree', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Save Best Model

In [ ]:
best_model = dt_model if best=='Decision Tree' else lr_model
model_data = {'model': best_model, 'encoders': encoders, 'feature_cols': list(X.columns), 'cat_cols': cat_cols}
with open('model.pkl','wb') as f:
    pickle.dump(model_data, f)
print(f'Best model ({best}) saved as model.pkl')

## 8. Test Prediction

In [ ]:
sample = {'Region':'Dodoma','Grape_Type':'Muscat','Season':'Dry Season',
          'Quality_Grade':'Grade A','Weight_kg':5.0,'Sugar_Content_Brix':20.5,
          'Farm_Size_ha':10.0,'Distance_to_Market_km':50.0,'Rainfall_mm':700.0}
with open('model.pkl','rb') as f:
    loaded = pickle.load(f)
sample_df = pd.DataFrame([sample])
for col in loaded['cat_cols']:
    sample_df[col] = loaded['encoders'][col].transform(sample_df[col])
pred = loaded['model'].predict(sample_df[loaded['feature_cols']])[0]
print(f'Predicted Grape Price: TZS {pred:,.2f} per kg')